# Basic PyTorch Classifier Live Build
Follow the instructor walkthrough to train a small neural network that predicts iris species. Seeds used for reproducibility: torch.manual_seed(42) before building the main model in step 9, and torch.manual_seed(0) for the fresh retraining in step 11.


## 1. Load the flowers
Read the 150-row CSV beside the notebook. Four measurements in centimeters are inputs; species is the three-class target.


In [1]:
import pandas as pd
flowers = pd.read_csv('iris_demo.csv')
print(flowers.head())
print(flowers.shape, flowers.dtypes)


   sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm species
0              5.1             3.5              1.4             0.2  setosa
1              4.9             3.0              1.4             0.2  setosa
2              4.7             3.2              1.3             0.2  setosa
3              4.6             3.1              1.5             0.2  setosa
4              5.0             3.6              1.4             0.2  setosa
(150, 5) sepal_length_cm    float64
sepal_width_cm     float64
petal_length_cm    float64
petal_width_cm     float64
species                str
dtype: object


## 2. Reserve the final test set
Set aside 20% before examining relationships or tuning. Stratification keeps each species represented.


In [2]:
from sklearn.model_selection import train_test_split
train_all, test = train_test_split(flowers, test_size=0.20, stratify=flowers['species'], random_state=42)
train, val = train_test_split(train_all, test_size=0.25, stratify=train_all['species'], random_state=42)
print(len(train), len(val), len(test))


90 30 30


## 3. Explore training data
Check missing values, ranges, labels, and a simple guess before fitting any model.


In [3]:
print(train.isna().sum())
print(train.describe())
print(train['species'].value_counts())
print(train.groupby('species')[['petal_length_cm','petal_width_cm']].median())
majority = train['species'].value_counts(normalize=True).max()
print('Majority baseline:', round(majority, 3))


sepal_length_cm    0
sepal_width_cm     0
petal_length_cm    0
petal_width_cm     0
species            0
dtype: int64
       sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm
count        90.000000       90.000000        90.000000       90.000000
mean          5.836667        3.013333         3.751111        1.197778
std           0.866149        0.416297         1.772083        0.761279
min           4.300000        2.000000         1.100000        0.100000
25%           5.100000        2.725000         1.600000        0.300000
50%           5.750000        3.000000         4.250000        1.300000
75%           6.400000        3.200000         5.100000        1.800000
max           7.900000        3.900000         6.900000        2.500000
species
virginica     30
setosa        30
versicolor    30
Name: count, dtype: int64
            petal_length_cm  petal_width_cm
species                                    
setosa                 1.50            0.20
versicolor       

## 4. See one tensor and one gradient
A tensor stores numbers; autograd supplies the slope needed to adjust a weight.


In [4]:
import torch
torch.manual_seed(42)
x = torch.tensor(3.0, requires_grad=True)
loss_example = (x - 1) ** 2
loss_example.backward()
print(loss_example.item(), x.grad.item())


4.0 4.0


## 5. Read a Linear layer
The first number is input width and the second is output width. Each output unit has four weights and one bias.


In [5]:
from torch import nn
probe = nn.Linear(4, 8)
print('weight shape:', probe.weight.shape)
print('bias shape:', probe.bias.shape)
print('parameter count:', sum(p.numel() for p in probe.parameters()))
one_flower = torch.tensor([2.0, 1.0, 0.5, -1.0])
print('one flower produces:', probe(one_flower).shape)


weight shape: torch.Size([8, 4])
bias shape: torch.Size([8])
parameter count: 40
one flower produces: torch.Size([8])


## 6. Compare activations
An activation changes each layer output before the next layer. Compare the same three input values.


In [6]:
z = torch.tensor([-2.0, 0.0, 2.0])
for name, activation in [('ReLU', nn.ReLU()), ('Leaky ReLU', nn.LeakyReLU(negative_slope=0.01)), ('sigmoid', nn.Sigmoid()), ('tanh', nn.Tanh())]:
    values = torch.round(activation(z) * 100) / 100
    print(name, values.tolist())


ReLU [0.0, 0.0, 2.0]
Leaky ReLU [-0.019999999552965164, 0.0, 2.0]
sigmoid [0.11999999731779099, 0.5, 0.8799999952316284]
tanh [-0.9599999785423279, 0.0, 0.9599999785423279]


## 7. Read the output scores
The final layer returns one raw score per class. CrossEntropyLoss expects these logits directly.


In [7]:
scores = torch.tensor([1.2, -0.4, 2.1])
print('predicted class ID:', scores.argmax().item())
print('illustrative probabilities:', torch.softmax(scores, dim=0))


predicted class ID: 2
illustrative probabilities: tensor([0.2731, 0.0551, 0.6717])


## 8. Prepare features and labels
Fit scaling on training rows only. CrossEntropyLoss expects integer class labels and raw logits.


In [8]:
from sklearn.preprocessing import StandardScaler
import numpy as np
features = ['sepal_length_cm','sepal_width_cm','petal_length_cm','petal_width_cm']
classes = sorted(train['species'].unique())
label_to_id = {name:i for i,name in enumerate(classes)}
scaler = StandardScaler().fit(train[features])
def as_tensors(frame):
    X = torch.tensor(scaler.transform(frame[features]), dtype=torch.float32)
    y = torch.tensor(frame['species'].map(label_to_id).to_numpy(), dtype=torch.long)
    return X, y
X_train, y_train = as_tensors(train)
X_val, y_val = as_tensors(val)
print(X_train.shape, y_train.shape, classes)


torch.Size([90, 4]) torch.Size([90]) ['setosa', 'versicolor', 'virginica']


## 9. Build and train a small network
Four inputs feed eight ReLU units and three output scores. The loss compares scores with labels. Seed 42 is set once right before building the model so the run is reproducible.


In [9]:
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(100):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
    if epoch in (0, 24, 49, 99):
        print(epoch + 1, round(loss.item(), 3))


1 1.042
25 0.044


50 0.184


100 0.021


## 10. Judge validation results
Use held-out validation rows to inspect accuracy and mistakes. No gradient is needed for evaluation.


In [10]:
from sklearn.metrics import accuracy_score, confusion_matrix
def evaluate(X, y):
    model.eval()
    with torch.inference_mode():
        pred = model(X).argmax(dim=1)
    return accuracy_score(y.numpy(), pred.numpy()), confusion_matrix(y.numpy(), pred.numpy(), labels=range(3))
val_acc, val_cm = evaluate(X_val, y_val)
print('Validation accuracy:', round(val_acc, 3))
print(val_cm)


Validation accuracy: 0.933
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


## 11. Make one controlled change
One controlled change only: hidden width goes from 8 to 4, learning rate stays at 0.01. Retrain from a fresh seed (0), compare validation accuracy with the original, and restore whichever model is better. On a tie the original width 8 is kept.


In [11]:
torch.manual_seed(0)
exp_model = nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Linear(4, 3))
exp_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)
exp_criterion = nn.CrossEntropyLoss()
exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=0.01)
for epoch in range(100):
    exp_model.train()
    for xb, yb in exp_loader:
        exp_optimizer.zero_grad()
        exp_logits = exp_model(xb)
        exp_loss = exp_criterion(exp_logits, yb)
        exp_loss.backward()
        exp_optimizer.step()
exp_model.eval()
with torch.inference_mode():
    exp_pred = exp_model(X_val).argmax(dim=1)
exp_acc = accuracy_score(y_val.numpy(), exp_pred.numpy())
print('Width 8 validation accuracy:', round(val_acc, 3))
print('Width 4 validation accuracy:', round(exp_acc, 3))
if exp_acc > val_acc:
    chosen_width = 4
    model = exp_model
    print('Keeping hidden width 4: it beat the original on validation.')
else:
    chosen_width = 8
    print('Keeping the original hidden width 8: it won or tied on validation.')


Width 8 validation accuracy: 0.933
Width 4 validation accuracy: 0.933
Keeping the original hidden width 8: it won or tied on validation.


## 12. Freeze choices and open the test set
After choosing the model with validation data, evaluate the untouched test set once.


In [12]:
X_test, y_test = as_tensors(test)
test_acc, test_cm = evaluate(X_test, y_test)
print('Final test accuracy:', round(test_acc, 3))
print(test_cm)


Final test accuracy: 0.933
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


## 13. Save and reload weights
Save learned parameters, rebuild the same architecture, load the data-only state, and enter evaluation mode.


In [13]:
torch.save(model.state_dict(), 'iris_pytorch_weights.pt')
reloaded = nn.Sequential(nn.Linear(4, chosen_width), nn.ReLU(), nn.Linear(chosen_width, 3))
reloaded.load_state_dict(torch.load('iris_pytorch_weights.pt', weights_only=True, map_location='cpu'))
reloaded.eval()
with torch.inference_mode():
    print(classes[reloaded(X_test[:1]).argmax(dim=1).item()])


setosa


## Exit ticket
Validation accuracy: 0.933. Final test accuracy: 0.933.
Reading the confusion matrix: setosa was never confused. The only mistakes were between versicolor and virginica, with one flower of each class predicted as the other. That fits what step 3 showed, where versicolor and virginica sit closer together in petal length and petal width than either sits to setosa.
The test set stayed closed while model choices were made so its score is an honest final check. If the test set had guided any choice, the model could have been tuned to that one test set and the score would no longer measure fresh, unseen data.
One likely source of error: versicolor and virginica overlap in petal measurements, so a flower with borderline petal size lands on the wrong side of the learned boundary.
One controlled next experiment: change only the learning rate to 0.005, retrain from a fresh seed, and compare validation accuracy against 0.933. With only 30 validation examples, one example is one accuracy step of about 0.033, so a small difference would not be meaningful on its own.
